In [3]:
import os
import io
import ast
import fitz
import json
import torch
import base64
import requests
import numpy as np
import pandas as pd
from PIL import Image
from dotenv import load_dotenv

from colpali_engine.models import ColQwen2, ColQwen2Processor

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.messages import HumanMessage, SystemMessage

In [9]:
load_dotenv()

True

In [5]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=7,
    check_every_n_seconds=0.1,
    max_bucket_size=10
)

In [7]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, rate_limiter=rate_limiter)

In [10]:
model_name = "vidore/colqwen2-v1.0"
model = ColQwen2.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
).eval()

Loading weights: 100%|██████████| 394/394 [00:00<00:00, 9235.35it/s]


In [11]:
processor = ColQwen2Processor.from_pretrained(
    model_name, 
    trust_remote_code=True
)

In [ ]:
def get_coarse_embedding(embeddings_tensor):
    pooled = embeddings_tensor.mean(dim=1) # mean pooling 
    normalised = pooled / pooled.norm(dim=-1, keepdim=True) # L2 normalization 
    return normalised.squeeze().to(torch.float32).cpu()

In [ ]:
def embed_query(query, model, processor):
    inputs = processor.process_queries(queries=[query])
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        embeddings = model(**inputs)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True) # L2 normalization

        coarse_vector = get_coarse_embedding(embeddings).flatten().tolist()
        embeddings = embeddings.squeeze(0).to(torch.float32).cpu()
    return coarse_vector, embeddings

: 